# Compare data structures

Two flavours of pairwise comparison:

1. **`compare()`** — point ratios with deterministic CI envelope from the summary's stored CIs. Fast, no samples needed.
2. **`bootstrap_ratio_ci()`** — percentile-bootstrap CI on raw per-iteration samples. Tighter and statistically rigorous.

In [ ]:
import glob
import kermit_lab as kl

kl.apply_style()
paths = sorted(glob.glob('../../bench-runs/*.json'))
df = kl.load(paths, criterion_root='../../target/criterion')
samples = kl.load_samples(paths, criterion_root='../../target/criterion')

## Pairwise speedup with envelope CIs

In [ ]:
iter_time = df[(df.metric == 'time') & (df.phase == 'iteration')]
result = kl.compare(iter_time, baseline='TreeTrie', target='ColumnTrie')
result[['query', 'tuples', 'speedup', 'speedup_lo', 'speedup_hi']]

## Bootstrap CI on raw samples (rigorous)

In [ ]:
labelled = samples.merge(
    df[['criterion_group', 'criterion_function', 'data_structure', 'phase']],
    on=['criterion_group', 'criterion_function'],
)
tt = labelled.query("data_structure == 'TreeTrie' and phase == 'iteration'").per_iter_ns.to_numpy()
ct = labelled.query("data_structure == 'ColumnTrie' and phase == 'iteration'").per_iter_ns.to_numpy()
lo, hi = kl.bootstrap_ratio_ci(tt, ct, rng=42)
print(f'TreeTrie / ColumnTrie speedup: {tt.mean()/ct.mean():.3f}  95% CI: [{lo:.3f}, {hi:.3f}]')